# EKF Robustness: Handling Sensor Failures

## Learning Objectives

- Handle sensor dropout (missing measurements)
- Estimate sensor bias using state augmentation
- Reject outlier measurements with chi-squared gating
- Diagnose EKF divergence using NIS analysis

## Why Robustness Matters

Real sensors fail in predictable ways. In production systems, you will encounter:

1. **Dropout**: Measurements go missing (GPS loss, sensor reboot, communication failure)
2. **Bias drift**: Sensors gradually drift from true values (temperature effects, aging)
3. **Outliers**: Sporadic bad measurements (multipath, electrical glitches, software bugs)

A naive EKF will diverge or produce poor estimates when these failures occur. This notebook teaches systematic handling of each failure mode using the simple pendulum system for clarity.

**Key insight**: The EKF framework naturally supports handling all these failures -- we just need to know the right techniques.

## Section 2: Setup

We import all necessary libraries and load the pendulum model. The `scipy.stats.chi2` distribution will be used later for outlier rejection thresholds.

In [ ]:
import numpy as np
import mujoco
import matplotlib.pyplot as plt
import mediapy as media
from scipy.stats import chi2
from scipy.optimize import approx_fprime
import ipywidgets
from ipywidgets import interact_manual, IntSlider, FloatSlider, FloatLogSlider

# Load the pendulum model
model = mujoco.MjModel.from_xml_path("../models/pendulum.xml")
data = mujoco.MjData(model)
renderer = mujoco.Renderer(model, height=480, width=640)

print(f"Model loaded: {model.nq} position DOF, {model.nv} velocity DOF")
print(f"Timestep: {model.opt.timestep} s")
print(f"Gravity: {model.opt.gravity}")
print(f"\nChi-squared thresholds for outlier detection:")
print(f"  1D: 95% = {chi2.ppf(0.95, df=1):.2f}, 99% = {chi2.ppf(0.99, df=1):.2f}")
print(f"  2D: 95% = {chi2.ppf(0.95, df=2):.2f}, 99% = {chi2.ppf(0.99, df=2):.2f}")

## Section 3: Baseline Pendulum EKF

First, we establish a baseline: a working EKF with perfect sensors (no failures). This gives us a reference to compare against when we introduce failures.

We reuse the EKF functions and pendulum dynamics from Notebook 01.

In [ ]:
# Physical parameters (must match pendulum.xml)
g = 9.81   # gravity (m/s^2)
L = 1.0    # pendulum length (m)
b = 0.1    # damping coefficient (matches <joint damping="0.1"/>)
dt = model.opt.timestep  # 0.002 s

# Noise covariance matrices
Q = np.diag([0.01, 0.1])  # Process noise: [position, velocity]
noise_std = 0.05  # Measurement noise (rad)
R = np.array([[noise_std**2]])  # R = [[0.0025]]

print(f"Physical parameters: g={g}, L={L}, b={b}, dt={dt}")
print(f"Process noise Q = diag([0.01, 0.1])")
print(f"Measurement noise R = [[{R[0,0]:.4f}]] (noise_std = {noise_std} rad)")

In [ ]:
# Pendulum dynamics functions
def pendulum_f(x, dt, g=9.81, L=1.0, b_damp=0.1):
    """
    State transition function: x_{k+1} = f(x_k, dt)
    
    x: (2,1) state vector [theta, omega]^T
    dt: time step (seconds)
    Returns: (2,1) predicted state
    """
    theta = x[0, 0]
    omega = x[1, 0]
    
    # Euler integration
    theta_new = theta + dt * omega
    omega_new = omega + dt * (-(g / L) * np.sin(theta) - b_damp * omega)
    
    return np.array([[theta_new], [omega_new]])


def pendulum_F(x, dt, g=9.81, L=1.0, b_damp=0.1):
    """
    Jacobian of f with respect to x: F = df/dx
    
    Returns: (2,2) Jacobian matrix
    """
    theta = x[0, 0]
    
    F = np.array([
        [1.0,                           dt],
        [-dt * (g / L) * np.cos(theta), 1.0 - dt * b_damp]
    ])
    return F


def measurement_h(x):
    """Measurement function: observe theta only."""
    return np.array([[x[0, 0]]])


def measurement_H(x):
    """Jacobian: dh/dx = [1, 0]."""
    return np.array([[1.0, 0.0]])


print("Dynamics functions defined: pendulum_f, pendulum_F, measurement_h, measurement_H")

In [ ]:
# EKF predict and update functions
def ekf_predict(x, P, f, F, Q, dt):
    """
    EKF Prediction Step.
    
    Returns: (x_pred, P_pred)
    """
    x_pred = f(x, dt)
    F_k = F(x, dt)
    P_pred = F_k @ P @ F_k.T + Q
    return x_pred, P_pred


def ekf_update(x, P, z, h, H, R):
    """
    EKF Update Step with Joseph form for numerical stability.
    
    Returns: (x_upd, P_upd)
    """
    H_k = H(x)
    y = z - h(x)  # Innovation
    S = H_k @ P @ H_k.T + R  # Innovation covariance
    K = P @ H_k.T @ np.linalg.inv(S)  # Kalman gain
    x_upd = x + K @ y
    
    # Joseph form for numerical stability
    I_KH = np.eye(x.shape[0]) - K @ H_k
    P_upd = I_KH @ P @ I_KH.T + K @ R @ K.T
    
    return x_upd, P_upd


print("EKF functions defined: ekf_predict, ekf_update")

In [ ]:
# Simulate pendulum ground truth
duration = 5.0  # seconds
initial_angle = 0.5  # rad

# Reset simulation
mujoco.mj_resetData(model, data)
data.qpos[0] = initial_angle
data.qvel[0] = 0.0
mujoco.mj_forward(model, data)

# Collect ground truth
true_states = []
times = []

while data.time < duration:
    true_states.append(np.array([[data.qpos[0].copy()], [data.qvel[0].copy()]]))
    times.append(data.time)
    mujoco.mj_step(model, data)

times = np.array(times)
n_steps = len(true_states)

print(f"Simulation complete: {n_steps} timesteps")
print(f"Time range: {times[0]:.3f} to {times[-1]:.3f} s")

In [ ]:
# Generate noisy measurements
np.random.seed(42)
measurements = []
for state in true_states:
    true_theta = state[0, 0]
    noisy_theta = true_theta + np.random.randn() * noise_std
    measurements.append(np.array([[noisy_theta]]))

print(f"Generated {len(measurements)} noisy measurements")
print(f"Measurement noise: {noise_std} rad ({np.degrees(noise_std):.1f} degrees)")

In [ ]:
# Run baseline EKF (everything working perfectly)
x_est = np.array([[0.0], [0.0]])  # Initial guess
P_est = np.eye(2) * 0.1  # Initial covariance

baseline_estimates = []
baseline_covariances = []

for k in range(n_steps):
    x_pred, P_pred = ekf_predict(x_est, P_est, pendulum_f, pendulum_F, Q, dt)
    x_est, P_est = ekf_update(x_pred, P_pred, measurements[k], measurement_h, measurement_H, R)
    baseline_estimates.append(x_est.copy())
    baseline_covariances.append(P_est.copy())

# Compute baseline RMSE
n_skip = int(0.5 / dt)  # Skip initial convergence
theta_errors = [abs(baseline_estimates[i][0,0] - true_states[i][0,0]) for i in range(n_skip, n_steps)]
baseline_rmse = np.sqrt(np.mean(np.array(theta_errors)**2))

print(f"Baseline EKF complete")
print(f"Position RMSE (after convergence): {baseline_rmse:.4f} rad")

In [ ]:
# Plot baseline results
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

true_theta = [s[0, 0] for s in true_states]
est_theta = [s[0, 0] for s in baseline_estimates]
sigma_theta = [2 * np.sqrt(P[0, 0]) for P in baseline_covariances]

# Position plot
axes[0].plot(times, true_theta, 'b-', lw=2, label='True')
axes[0].plot(times, est_theta, 'r--', lw=2, label='EKF Estimate')
step = max(1, n_steps // 200)
meas_theta = [m[0, 0] for m in measurements]
axes[0].scatter(times[::step], meas_theta[::step], c='g', s=10, alpha=0.4, label='Measurements')
axes[0].fill_between(times, 
                     [est_theta[i] - sigma_theta[i] for i in range(n_steps)],
                     [est_theta[i] + sigma_theta[i] for i in range(n_steps)],
                     color='red', alpha=0.1, label='2-sigma')
axes[0].set_ylabel('Angle (rad)')
axes[0].set_title('Baseline EKF: Everything Working')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# Covariance plot
axes[1].semilogy(times, [P[0, 0] for P in baseline_covariances], 'b-', lw=2, label='P[0,0] (position)')
axes[1].semilogy(times, [P[1, 1] for P in baseline_covariances], 'r-', lw=2, label='P[1,1] (velocity)')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Variance (log scale)')
axes[1].set_title('Covariance Diagonal: Converges to Steady State')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nBaseline RMSE: {baseline_rmse:.4f} rad")
print("This is our reference for 'everything working correctly'.")

## Section 4: Sensor Dropout Handling

**Problem**: The sensor goes offline for some period -- no measurements available.

**Solution**: Skip the update step when measurements are missing. The EKF naturally handles this!

**Key insight**: During dropout:
- The **predict step** continues using the dynamics model
- The **update step** is skipped (no measurement to incorporate)
- **Covariance grows** because we lose information from measurements
- When measurements resume, the filter **corrects and recovers**

In [ ]:
def ekf_step_with_dropout(x_est, P_est, measurement_available, z, f, F, Q, h, H, R, dt):
    """
    EKF step that handles missing measurements.
    
    Parameters:
        x_est: (n,1) current state estimate
        P_est: (n,n) current covariance
        measurement_available: bool, True if measurement z is valid
        z: (m,1) measurement (may be None if not available)
        f, F: dynamics function and Jacobian
        Q: (n,n) process noise covariance
        h, H: measurement function and Jacobian
        R: (m,m) measurement noise covariance
        dt: time step
        
    Returns: (x_est, P_est) updated state and covariance
    """
    # Always predict
    x_pred, P_pred = ekf_predict(x_est, P_est, f, F, Q, dt)
    
    # Update only if measurement available
    if measurement_available:
        x_est, P_est = ekf_update(x_pred, P_pred, z, h, H, R)
    else:
        # No update - covariance grows, estimate relies on model only
        x_est, P_est = x_pred, P_pred
    
    return x_est, P_est


print("ekf_step_with_dropout function defined")
print("  - Always performs predict step")
print("  - Skips update if measurement_available=False")
print("  - Covariance grows during dropout (more uncertainty)")

In [ ]:
# Demonstrate dropout: measurements unavailable from timestep 1000 to 1500
dropout_start = 1000
dropout_end = 1500
dropout_duration = dropout_end - dropout_start
dropout_time_start = dropout_start * dt
dropout_time_end = dropout_end * dt

print(f"Dropout period: timesteps {dropout_start}-{dropout_end}")
print(f"Dropout duration: {dropout_duration * dt:.2f} seconds ({dropout_time_start:.2f}s to {dropout_time_end:.2f}s)")

In [ ]:
# Run EKF with dropout
x_est = np.array([[0.0], [0.0]])
P_est = np.eye(2) * 0.1

dropout_estimates = []
dropout_covariances = []

for k in range(n_steps):
    # Check if measurement is available
    measurement_available = not (dropout_start <= k < dropout_end)
    z = measurements[k] if measurement_available else None
    
    x_est, P_est = ekf_step_with_dropout(
        x_est, P_est, measurement_available, z,
        pendulum_f, pendulum_F, Q,
        measurement_h, measurement_H, R, dt
    )
    
    dropout_estimates.append(x_est.copy())
    dropout_covariances.append(P_est.copy())

print("EKF with dropout complete")

In [ ]:
# Plot dropout results
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

true_theta = [s[0, 0] for s in true_states]
dropout_theta = [s[0, 0] for s in dropout_estimates]
sigma_theta = [2 * np.sqrt(P[0, 0]) for P in dropout_covariances]

# 1. Estimate vs true with dropout region
axes[0].plot(times, true_theta, 'b-', lw=2, label='True')
axes[0].plot(times, dropout_theta, 'r--', lw=2, label='EKF Estimate')
axes[0].axvspan(dropout_time_start, dropout_time_end, alpha=0.2, color='gray', label='Dropout region')
axes[0].fill_between(times,
                     [dropout_theta[i] - sigma_theta[i] for i in range(n_steps)],
                     [dropout_theta[i] + sigma_theta[i] for i in range(n_steps)],
                     color='red', alpha=0.1, label='2-sigma')
axes[0].set_ylabel('Angle (rad)')
axes[0].set_title('Estimate vs True (gray = dropout region)')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# 2. Covariance diagonal over time
axes[1].semilogy(times, [P[0, 0] for P in dropout_covariances], 'b-', lw=2, label='P[0,0] (position)')
axes[1].semilogy(times, [P[1, 1] for P in dropout_covariances], 'r-', lw=2, label='P[1,1] (velocity)')
axes[1].axvspan(dropout_time_start, dropout_time_end, alpha=0.2, color='gray', label='Dropout region')
axes[1].set_ylabel('Variance (log scale)')
axes[1].set_title('Covariance Growth During Dropout')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

# 3. Estimation error with 2-sigma bounds
errors = [dropout_theta[i] - true_theta[i] for i in range(n_steps)]
axes[2].plot(times, errors, 'b-', lw=1.5, label='Error')
axes[2].fill_between(times, [-s for s in sigma_theta], sigma_theta, color='red', alpha=0.2, label='2-sigma bounds')
axes[2].axvspan(dropout_time_start, dropout_time_end, alpha=0.2, color='gray', label='Dropout region')
axes[2].axhline(0, color='k', linestyle='-', lw=0.5)
axes[2].set_xlabel('Time (s)')
axes[2].set_ylabel('Error (rad)')
axes[2].set_title('Estimation Error vs 2-sigma Bounds')
axes[2].legend(loc='upper right')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Dropout Takeaways

**What we observe:**
1. **Covariance grows** during dropout -- the filter knows it's losing information
2. **Estimate drifts** from true state -- model alone isn't perfect
3. **Error stays within bounds** -- the growing covariance captures increasing uncertainty
4. **Recovery happens automatically** -- when measurements resume, covariance shrinks and estimate corrects

**Key insight**: Longer dropout = larger uncertainty = bigger correction jump when measurements resume.

## Section 5: Interactive Dropout Exploration

Use the sliders to explore how dropout timing and duration affect the EKF behavior.

In [ ]:
def run_ekf_with_dropout_params(dropout_start, dropout_duration):
    """
    Run EKF with configurable dropout parameters.
    """
    dropout_end = dropout_start + dropout_duration
    
    x_est = np.array([[0.0], [0.0]])
    P_est = np.eye(2) * 0.1
    
    estimates = []
    covariances = []
    
    for k in range(n_steps):
        measurement_available = not (dropout_start <= k < dropout_end)
        z = measurements[k] if measurement_available else None
        
        x_est, P_est = ekf_step_with_dropout(
            x_est, P_est, measurement_available, z,
            pendulum_f, pendulum_F, Q,
            measurement_h, measurement_H, R, dt
        )
        
        estimates.append(x_est.copy())
        covariances.append(P_est.copy())
    
    return estimates, covariances, dropout_end


@interact_manual(
    dropout_start=IntSlider(value=1000, min=0, max=2000, step=50, description='Start:'),
    dropout_duration=IntSlider(value=500, min=0, max=1000, step=50, description='Duration:')
)
def explore_dropout(dropout_start, dropout_duration):
    """Explore effect of dropout timing and duration."""
    estimates, covariances, dropout_end = run_ekf_with_dropout_params(dropout_start, dropout_duration)
    
    dropout_time_start = dropout_start * dt
    dropout_time_end = dropout_end * dt
    
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
    
    true_theta = [s[0, 0] for s in true_states]
    est_theta = [s[0, 0] for s in estimates]
    sigma_theta = [2 * np.sqrt(P[0, 0]) for P in covariances]
    
    # Estimate plot
    axes[0].plot(times, true_theta, 'b-', lw=2, label='True')
    axes[0].plot(times, est_theta, 'r--', lw=2, label='EKF Estimate')
    if dropout_duration > 0:
        axes[0].axvspan(dropout_time_start, dropout_time_end, alpha=0.2, color='gray', label='Dropout')
    axes[0].fill_between(times,
                         [est_theta[i] - sigma_theta[i] for i in range(n_steps)],
                         [est_theta[i] + sigma_theta[i] for i in range(n_steps)],
                         color='red', alpha=0.1, label='2-sigma')
    axes[0].set_ylabel('Angle (rad)')
    axes[0].set_title(f'Dropout from {dropout_time_start:.2f}s to {dropout_time_end:.2f}s ({dropout_duration * dt:.2f}s duration)')
    axes[0].legend(loc='upper right')
    axes[0].grid(True, alpha=0.3)
    
    # Covariance plot
    axes[1].semilogy(times, [P[0, 0] for P in covariances], 'b-', lw=2, label='P[0,0] (position)')
    axes[1].semilogy(times, [P[1, 1] for P in covariances], 'r-', lw=2, label='P[1,1] (velocity)')
    if dropout_duration > 0:
        axes[1].axvspan(dropout_time_start, dropout_time_end, alpha=0.2, color='gray', label='Dropout')
    axes[1].set_xlabel('Time (s)')
    axes[1].set_ylabel('Variance (log scale)')
    axes[1].set_title('Covariance Growth and Recovery')
    axes[1].legend(loc='upper right')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Compute max covariance during dropout
    if dropout_duration > 0:
        max_P00_during_dropout = max([covariances[k][0,0] for k in range(dropout_start, min(dropout_end, n_steps))])
        print(f"Max position variance during dropout: {max_P00_during_dropout:.4f}")
        print(f"Max position 2-sigma: {2 * np.sqrt(max_P00_during_dropout):.4f} rad")
    
    print("\nTry different durations to see:")
    print("  - Longer dropout -> larger covariance growth")
    print("  - Longer dropout -> larger correction jump when measurements resume")
    print("  - Position matters: dropout during high velocity causes more drift")

## Section 6: Sensor Bias and State Augmentation

**Problem**: The sensor has a slow-varying bias that corrupts measurements. The true value is `theta`, but we measure `theta + bias`.

**Solution**: Augment the state vector to include the bias as an additional state to estimate.

- Original state: $\mathbf{x} = \begin{bmatrix} \theta \\ \omega \end{bmatrix}$ (2,1)
- Augmented state: $\mathbf{x}_{aug} = \begin{bmatrix} \theta \\ \omega \\ b_\theta \end{bmatrix}$ (3,1)

**Bias model**: We model the bias as a **random walk** -- it stays roughly constant but drifts slowly over time. The process noise on the bias state controls how fast the EKF thinks the bias can change.

**Why this works**: The EKF can distinguish bias from the true state because the dynamics behave differently:
- True angle changes according to physics (pendulum equation)
- Bias is approximately constant (just drifts slowly)

This observability comes from velocity -- if the angle measurement changes but velocity dynamics don't match, the EKF attributes the discrepancy to bias.

In [ ]:
# Augmented state dynamics functions
def pendulum_f_augmented(x_aug, dt, g=9.81, L=1.0, b_damp=0.1):
    """
    State transition for pendulum with bias state.
    
    x_aug: (3,1) state [theta, omega, b_theta]^T
    Returns: (3,1) predicted state
    """
    theta = x_aug[0, 0]
    omega = x_aug[1, 0]
    bias = x_aug[2, 0]
    
    # Original dynamics (unchanged)
    theta_new = theta + dt * omega
    omega_new = omega + dt * (-(g / L) * np.sin(theta) - b_damp * omega)
    # Bias: random walk model (constant, process noise adds drift)
    bias_new = bias
    
    return np.array([[theta_new], [omega_new], [bias_new]])


def pendulum_F_augmented(x_aug, dt, g=9.81, L=1.0, b_damp=0.1):
    """
    Jacobian for augmented system.
    
    Returns: (3,3) Jacobian matrix
    """
    theta = x_aug[0, 0]
    F = np.array([
        [1.0,                         dt,               0.0],
        [-dt * (g / L) * np.cos(theta), 1.0 - dt * b_damp, 0.0],
        [0.0,                         0.0,              1.0]
    ])
    return F


def measurement_h_biased(x_aug):
    """
    Measurement with bias: z = theta + b.
    
    x_aug: (3,1) state [theta, omega, b_theta]^T
    Returns: (1,1) measurement
    """
    theta = x_aug[0, 0]
    bias = x_aug[2, 0]
    return np.array([[theta + bias]])


def measurement_H_biased(x_aug):
    """
    Jacobian: dz/d[theta, omega, b] = [1, 0, 1].
    
    Returns: (1,3) Jacobian matrix
    """
    return np.array([[1.0, 0.0, 1.0]])


print("Augmented dynamics functions defined:")
print("  pendulum_f_augmented: (3,1) -> (3,1) state transition")
print("  pendulum_F_augmented: (3,1) -> (3,3) Jacobian")
print("  measurement_h_biased: z = theta + bias")
print("  measurement_H_biased: H = [1, 0, 1]")

In [ ]:
# Verify augmented Jacobians against finite differences
def verify_jacobian_augmented(f, F, x, dt, eps=1e-7):
    """Verify analytical Jacobian F against finite differences."""
    n = x.shape[0]
    F_analytical = F(x, dt)
    x_flat = x.flatten()
    F_numerical = np.zeros((n, n))
    
    for i in range(n):
        def f_i(x_flat_in, idx=i):
            x_col = x_flat_in.reshape(-1, 1)
            return f(x_col, dt)[idx, 0]
        F_numerical[i, :] = approx_fprime(x_flat, f_i, eps)
    
    error = np.max(np.abs(F_analytical - F_numerical))
    return error


# Test at a few states
test_states_aug = [
    np.array([[0.5], [0.0], [0.0]]),   # theta=0.5, no bias
    np.array([[0.3], [1.0], [0.05]]),  # with velocity and bias
    np.array([[np.pi/4], [-0.5], [0.1]]),  # 45 degrees with bias
]

print("Augmented Jacobian Verification:")
for x_test in test_states_aug:
    error = verify_jacobian_augmented(pendulum_f_augmented, pendulum_F_augmented, x_test, dt)
    status = "PASS" if error < 1e-5 else "FAIL"
    print(f"  x = {x_test.T.tolist()[0]} -> error = {error:.2e} [{status}]")

print("\nAll Jacobians verified.")

## Section 7: Bias Estimation Demo

We simulate a scenario where the sensor has a linearly drifting bias:
- Bias starts at 0 and grows to 0.1 rad over 5 seconds
- Measurements: $z_k = \theta_{true,k} + bias_k + noise$

We run two EKFs:
1. **Non-augmented** (ignores bias) -- will drift with the sensor
2. **Augmented** (estimates bias) -- should learn the bias and correct for it

In [ ]:
# Generate biased measurements
np.random.seed(42)
final_bias = 0.1  # rad (about 6 degrees)

true_bias = [final_bias * (k / n_steps) for k in range(n_steps)]  # Linear drift
biased_measurements = []

for k, state in enumerate(true_states):
    true_theta = state[0, 0]
    z = true_theta + true_bias[k] + np.random.randn() * noise_std
    biased_measurements.append(np.array([[z]]))

print(f"Biased measurements generated")
print(f"Bias drift: 0 -> {final_bias} rad over {duration}s")
print(f"Bias rate: {final_bias/duration:.4f} rad/s ({np.degrees(final_bias/duration):.2f} deg/s)")

In [ ]:
# Run non-augmented EKF (ignores bias)
x_est = np.array([[0.0], [0.0]])
P_est = np.eye(2) * 0.1

nonaug_estimates = []

for k in range(n_steps):
    x_pred, P_pred = ekf_predict(x_est, P_est, pendulum_f, pendulum_F, Q, dt)
    x_est, P_est = ekf_update(x_pred, P_pred, biased_measurements[k],
                               measurement_h, measurement_H, R)
    nonaug_estimates.append(x_est.copy())

print("Non-augmented EKF complete (ignores bias)")

In [ ]:
# Run augmented EKF (estimates bias)
x_est_aug = np.array([[0.0], [0.0], [0.0]])  # Initial bias estimate = 0
P_est_aug = np.diag([0.1, 0.1, 0.01])  # Small initial bias uncertainty

# Process noise for augmented system
Q_aug = np.diag([0.01, 0.1, 1e-6])  # Tiny process noise on bias (slow drift)

aug_estimates = []
aug_covariances = []

for k in range(n_steps):
    # Predict
    x_pred_aug = pendulum_f_augmented(x_est_aug, dt)
    F_aug = pendulum_F_augmented(x_est_aug, dt)
    P_pred_aug = F_aug @ P_est_aug @ F_aug.T + Q_aug
    
    # Update
    H_aug = measurement_H_biased(x_pred_aug)
    y = biased_measurements[k] - measurement_h_biased(x_pred_aug)
    S = H_aug @ P_pred_aug @ H_aug.T + R
    K = P_pred_aug @ H_aug.T @ np.linalg.inv(S)
    x_est_aug = x_pred_aug + K @ y
    
    # Joseph form
    I_KH = np.eye(3) - K @ H_aug
    P_est_aug = I_KH @ P_pred_aug @ I_KH.T + K @ R @ K.T
    
    aug_estimates.append(x_est_aug.copy())
    aug_covariances.append(P_est_aug.copy())

print("Augmented EKF complete (estimates bias)")
print(f"Final estimated bias: {aug_estimates[-1][2,0]:.4f} rad")
print(f"Final true bias: {true_bias[-1]:.4f} rad")
print(f"Bias estimation error: {abs(aug_estimates[-1][2,0] - true_bias[-1]):.4f} rad")

In [ ]:
# Plot bias estimation results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

true_theta = [s[0, 0] for s in true_states]
nonaug_theta = [s[0, 0] for s in nonaug_estimates]
aug_theta = [s[0, 0] for s in aug_estimates]
est_bias = [s[2, 0] for s in aug_estimates]

# 1. True vs estimated theta (augmented EKF)
axes[0, 0].plot(times, true_theta, 'b-', lw=2, label='True theta')
axes[0, 0].plot(times, aug_theta, 'r--', lw=2, label='Augmented EKF')
axes[0, 0].set_ylabel('Angle (rad)')
axes[0, 0].set_title('Augmented EKF: Tracks True Theta')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. True bias vs estimated bias
sigma_bias = [2 * np.sqrt(P[2, 2]) for P in aug_covariances]
axes[0, 1].plot(times, true_bias, 'b-', lw=2, label='True bias')
axes[0, 1].plot(times, est_bias, 'r--', lw=2, label='Estimated bias')
axes[0, 1].fill_between(times,
                        [est_bias[i] - sigma_bias[i] for i in range(n_steps)],
                        [est_bias[i] + sigma_bias[i] for i in range(n_steps)],
                        color='red', alpha=0.1, label='2-sigma')
axes[0, 1].set_ylabel('Bias (rad)')
axes[0, 1].set_title('EKF Learns the Drifting Bias!')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Bias estimation error
bias_errors = [est_bias[i] - true_bias[i] for i in range(n_steps)]
axes[1, 0].plot(times, bias_errors, 'b-', lw=1.5, label='Bias error')
axes[1, 0].fill_between(times, [-s for s in sigma_bias], sigma_bias, 
                        color='red', alpha=0.2, label='2-sigma bounds')
axes[1, 0].axhline(0, color='k', linestyle='-', lw=0.5)
axes[1, 0].set_xlabel('Time (s)')
axes[1, 0].set_ylabel('Error (rad)')
axes[1, 0].set_title('Bias Estimation Error')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Comparison: augmented vs non-augmented
axes[1, 1].plot(times, true_theta, 'b-', lw=2, label='True theta')
axes[1, 1].plot(times, nonaug_theta, 'g--', lw=2, label='Non-augmented (drifts with bias)')
axes[1, 1].plot(times, aug_theta, 'r--', lw=2, label='Augmented (corrects for bias)')
axes[1, 1].set_xlabel('Time (s)')
axes[1, 1].set_ylabel('Angle (rad)')
axes[1, 1].set_title('Comparison: Augmented vs Non-Augmented')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print RMSE comparison
nonaug_errors = [abs(nonaug_theta[i] - true_theta[i]) for i in range(n_skip, n_steps)]
aug_errors = [abs(aug_theta[i] - true_theta[i]) for i in range(n_skip, n_steps)]

print(f"\nPosition RMSE (after convergence):")
print(f"  Non-augmented: {np.sqrt(np.mean(np.array(nonaug_errors)**2)):.4f} rad")
print(f"  Augmented:     {np.sqrt(np.mean(np.array(aug_errors)**2)):.4f} rad")

### Bias Estimation Takeaways

**What we observe:**
1. **Non-augmented EKF drifts** with the sensor bias -- it thinks the measurement IS the true angle
2. **Augmented EKF learns the bias** -- it separates true state from sensor error
3. **Bias estimate converges** -- starts at 0, tracks the drifting true bias

**Key insight**: State augmentation lets EKF learn unknown parameters. The bias must be **observable** -- dynamics must distinguish bias from true state. For the pendulum, velocity provides this observability: if measurements change but velocity dynamics don't match, the EKF attributes the discrepancy to bias.

**Practical uses:**
- Gyroscope bias estimation (very common in IMU navigation)
- Accelerometer bias estimation
- Magnetometer calibration
- Any slow-varying sensor offset

## Section 8: Outlier Detection and Rejection

**Problem**: Sporadic bad measurements (multipath GPS, electrical glitches, sensor failures) can corrupt the estimate.

**Solution**: Use **Normalized Innovation Squared (NIS)** to detect outliers and reject them.

### What is NIS?

The innovation is the difference between the measurement and what we predicted:
$$\mathbf{y} = \mathbf{z} - h(\mathbf{x}_{pred})$$

The **Normalized Innovation Squared (NIS)** weights this by the innovation covariance:
$$\text{NIS} = \mathbf{y}^T \mathbf{S}^{-1} \mathbf{y}$$
where $\mathbf{S} = \mathbf{H} \mathbf{P}_{pred} \mathbf{H}^T + \mathbf{R}$

**Key insight**: For a consistent filter, NIS is **chi-squared distributed** with degrees of freedom equal to the measurement dimension. Large NIS means the measurement is inconsistent with the predicted state -- likely an outlier!

**Gating rule**: Reject measurement if NIS > threshold, where threshold is the chi-squared percentile (e.g., 99%).

In [ ]:
def compute_nis(z, x_pred, h, H, P_pred, R):
    """
    Compute Normalized Innovation Squared (NIS).
    
    NIS = (z - h(x))^T @ S^(-1) @ (z - h(x))
    where S = H @ P @ H^T + R
    
    Parameters:
        z: (m,1) measurement
        x_pred: (n,1) predicted state
        h: measurement function h(x) -> (m,1)
        H: Jacobian function H(x) -> (m,n)
        P_pred: (n,n) predicted covariance
        R: (m,m) measurement noise covariance
        
    Returns: scalar NIS value
    """
    H_k = H(x_pred)
    innovation = z - h(x_pred)
    S = H_k @ P_pred @ H_k.T + R
    nis = float(innovation.T @ np.linalg.inv(S) @ innovation)
    return nis


def get_gate_threshold(measurement_dim, confidence=0.99):
    """
    Get chi-squared threshold for given confidence level.
    
    Parameters:
        measurement_dim: number of measurements (chi-squared degrees of freedom)
        confidence: probability that valid measurement has NIS below threshold
        
    Returns: threshold value
    """
    return chi2.ppf(confidence, df=measurement_dim)


def ekf_update_with_gating(x_pred, P_pred, z, h, H, R, gate_threshold):
    """
    EKF update with chi-squared gating for outlier rejection.
    
    Returns: (x_upd, P_upd, is_accepted)
    """
    nis = compute_nis(z, x_pred, h, H, P_pred, R)
    
    if nis > gate_threshold:
        # Reject outlier - skip update
        return x_pred, P_pred, False
    else:
        # Accept - normal update
        x_upd, P_upd = ekf_update(x_pred, P_pred, z, h, H, R)
        return x_upd, P_upd, True


# Print chi-squared thresholds for reference
print("Chi-squared gating functions defined:")
print("  compute_nis: Calculate Normalized Innovation Squared")
print("  get_gate_threshold: Get chi-squared threshold from scipy.stats")
print("  ekf_update_with_gating: Update with outlier rejection")
print("")
print("Chi-squared thresholds:")
print(f"  1D: 95% = {chi2.ppf(0.95, df=1):.2f}, 99% = {chi2.ppf(0.99, df=1):.2f}")
print(f"  2D: 95% = {chi2.ppf(0.95, df=2):.2f}, 99% = {chi2.ppf(0.99, df=2):.2f}")

## Section 9: Outlier Rejection Demo

We inject outliers at specific timesteps and compare:
1. **Naive EKF** (no gating) -- jumps on outliers
2. **Gated EKF** (chi-squared rejection) -- rejects outliers and stays smooth

In [ ]:
# Create measurements with outliers
np.random.seed(42)

# Outlier parameters
outlier_indices = [500, 1000, 1500, 2000]
outlier_magnitude = 10 * np.sqrt(R[0, 0])  # 10 sigma

# Generate clean measurements
outlier_measurements = []
for k, state in enumerate(true_states):
    true_theta = state[0, 0]
    z = true_theta + np.random.randn() * noise_std
    
    # Inject outlier
    if k in outlier_indices:
        z += outlier_magnitude
    
    outlier_measurements.append(np.array([[z]]))

print(f"Outliers injected at timesteps: {outlier_indices}")
print(f"Outlier times: {[idx * dt for idx in outlier_indices]} s")
print(f"Outlier magnitude: {outlier_magnitude:.3f} rad ({outlier_magnitude/noise_std:.0f} sigma)")

In [ ]:
# Run naive EKF (no gating)
x_est = np.array([[0.0], [0.0]])
P_est = np.eye(2) * 0.1

naive_estimates = []
naive_nis = []

for k in range(n_steps):
    x_pred, P_pred = ekf_predict(x_est, P_est, pendulum_f, pendulum_F, Q, dt)
    
    # Compute NIS for analysis
    nis = compute_nis(outlier_measurements[k], x_pred, measurement_h, measurement_H, P_pred, R)
    naive_nis.append(nis)
    
    # Always update (no rejection)
    x_est, P_est = ekf_update(x_pred, P_pred, outlier_measurements[k],
                               measurement_h, measurement_H, R)
    naive_estimates.append(x_est.copy())

print("Naive EKF (no outlier rejection) complete")

In [ ]:
# Run gated EKF (with chi-squared rejection)
x_est = np.array([[0.0], [0.0]])
P_est = np.eye(2) * 0.1

gate_threshold = get_gate_threshold(measurement_dim=1, confidence=0.99)

gated_estimates = []
gated_nis = []
rejected_indices = []

for k in range(n_steps):
    x_pred, P_pred = ekf_predict(x_est, P_est, pendulum_f, pendulum_F, Q, dt)
    
    # Compute NIS
    nis = compute_nis(outlier_measurements[k], x_pred, measurement_h, measurement_H, P_pred, R)
    gated_nis.append(nis)
    
    # Update with gating
    x_est, P_est, accepted = ekf_update_with_gating(
        x_pred, P_pred, outlier_measurements[k],
        measurement_h, measurement_H, R, gate_threshold
    )
    
    if not accepted:
        rejected_indices.append(k)
    
    gated_estimates.append(x_est.copy())

print(f"Gated EKF (with outlier rejection) complete")
print(f"Gate threshold (99%): {gate_threshold:.2f}")
print(f"Rejected {len(rejected_indices)} measurements at indices: {rejected_indices}")
print(f"Expected rejections: {len(outlier_indices)} (injected outliers)")

In [ ]:
# Plot outlier rejection results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

true_theta = [s[0, 0] for s in true_states]
naive_theta = [s[0, 0] for s in naive_estimates]
gated_theta = [s[0, 0] for s in gated_estimates]
meas_theta = [m[0, 0] for m in outlier_measurements]

# 1. Measurements with outliers highlighted
axes[0, 0].plot(times, meas_theta, 'g-', lw=0.5, alpha=0.5, label='Measurements')
axes[0, 0].plot(times, true_theta, 'b-', lw=2, label='True')
for idx in outlier_indices:
    axes[0, 0].scatter(times[idx], meas_theta[idx], c='red', s=100, marker='x', zorder=5)
axes[0, 0].scatter([], [], c='red', s=100, marker='x', label='Outliers')  # Legend entry
axes[0, 0].set_ylabel('Angle (rad)')
axes[0, 0].set_title('Measurements with Injected Outliers')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Naive EKF (jumps on outliers)
axes[0, 1].plot(times, true_theta, 'b-', lw=2, label='True')
axes[0, 1].plot(times, naive_theta, 'r--', lw=2, label='Naive EKF')
for idx in outlier_indices:
    axes[0, 1].axvline(times[idx], color='red', linestyle=':', alpha=0.5)
axes[0, 1].set_ylabel('Angle (rad)')
axes[0, 1].set_title('Naive EKF: Jumps on Outliers')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Gated EKF (smooth, outliers rejected)
axes[1, 0].plot(times, true_theta, 'b-', lw=2, label='True')
axes[1, 0].plot(times, gated_theta, 'r--', lw=2, label='Gated EKF')
for idx in outlier_indices:
    axes[1, 0].axvline(times[idx], color='gray', linestyle=':', alpha=0.5)
axes[1, 0].set_xlabel('Time (s)')
axes[1, 0].set_ylabel('Angle (rad)')
axes[1, 0].set_title('Gated EKF: Outliers Rejected, Smooth Tracking')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. NIS time series with threshold
axes[1, 1].semilogy(times, gated_nis, 'b-', lw=0.5, alpha=0.7, label='NIS')
axes[1, 1].axhline(gate_threshold, color='red', linestyle='--', lw=2, label=f'99% threshold ({gate_threshold:.2f})')
axes[1, 1].axhline(1.0, color='green', linestyle='-', lw=1, label='Expected mean (1.0)')
for idx in outlier_indices:
    axes[1, 1].scatter(times[idx], gated_nis[idx], c='red', s=100, marker='x', zorder=5)
axes[1, 1].set_xlabel('Time (s)')
axes[1, 1].set_ylabel('NIS (log scale)')
axes[1, 1].set_title('NIS Time Series: Outliers Spike Above Threshold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_ylim([0.01, 1000])

plt.tight_layout()
plt.show()

# Print rejection statistics
print(f"\nRejection Statistics:")
print(f"  Outliers injected: {len(outlier_indices)} at timesteps {outlier_indices}")
print(f"  Measurements rejected: {len(rejected_indices)} at timesteps {rejected_indices}")
print(f"  Match: {'Yes' if set(outlier_indices) == set(rejected_indices) else 'No'}")

# RMSE comparison
naive_errors = [abs(naive_theta[i] - true_theta[i]) for i in range(n_skip, n_steps)]
gated_errors = [abs(gated_theta[i] - true_theta[i]) for i in range(n_skip, n_steps)]
print(f"\nPosition RMSE:")
print(f"  Naive EKF:  {np.sqrt(np.mean(np.array(naive_errors)**2)):.4f} rad")
print(f"  Gated EKF:  {np.sqrt(np.mean(np.array(gated_errors)**2)):.4f} rad")

### Gate Threshold Selection

**Trade-off**: The gate threshold balances two errors:
1. **Too aggressive (low threshold)**: Rejects valid but noisy measurements, estimate drifts
2. **Too loose (high threshold)**: Accepts outliers, estimate corrupted

**Rule of thumb**:
- **95% confidence**: Rejects 5% of valid measurements (aggressive)
- **99% confidence**: Rejects 1% of valid measurements (conservative, recommended)
- **99.5% confidence**: Very conservative, outliers must be extreme

## Section 10: Interactive Gating Exploration

Explore how confidence level, outlier magnitude, and number of outliers affect gating performance.

In [ ]:
def run_ekf_with_gating_params(confidence_level, outlier_magnitude_sigma, n_outliers):
    """
    Run EKF with configurable gating parameters.
    """
    np.random.seed(42)
    
    # Generate outlier indices (evenly spaced)
    if n_outliers > 0:
        outlier_step = n_steps // (n_outliers + 1)
        outlier_idx = [outlier_step * (i + 1) for i in range(n_outliers)]
    else:
        outlier_idx = []
    
    # Generate measurements with outliers
    meas = []
    for k, state in enumerate(true_states):
        true_theta = state[0, 0]
        z = true_theta + np.random.randn() * noise_std
        if k in outlier_idx:
            z += outlier_magnitude_sigma * noise_std
        meas.append(np.array([[z]]))
    
    # Run gated EKF
    gate_thresh = get_gate_threshold(measurement_dim=1, confidence=confidence_level)
    
    x_est = np.array([[0.0], [0.0]])
    P_est = np.eye(2) * 0.1
    
    estimates = []
    nis_vals = []
    rejected = []
    
    for k in range(n_steps):
        x_pred, P_pred = ekf_predict(x_est, P_est, pendulum_f, pendulum_F, Q, dt)
        nis = compute_nis(meas[k], x_pred, measurement_h, measurement_H, P_pred, R)
        nis_vals.append(nis)
        
        x_est, P_est, accepted = ekf_update_with_gating(
            x_pred, P_pred, meas[k],
            measurement_h, measurement_H, R, gate_thresh
        )
        if not accepted:
            rejected.append(k)
        estimates.append(x_est.copy())
    
    return estimates, nis_vals, rejected, outlier_idx, gate_thresh, meas


@interact_manual(
    confidence_level=FloatSlider(value=0.99, min=0.90, max=0.999, step=0.005, description='Confidence:'),
    outlier_magnitude_sigma=FloatSlider(value=10.0, min=0.0, max=20.0, step=1.0, description='Outlier (sigma):'),
    n_outliers=IntSlider(value=4, min=0, max=10, step=1, description='N outliers:')
)
def explore_gating(confidence_level, outlier_magnitude_sigma, n_outliers):
    """Explore effect of gating parameters."""
    estimates, nis_vals, rejected, outlier_idx, gate_thresh, meas = run_ekf_with_gating_params(
        confidence_level, outlier_magnitude_sigma, n_outliers
    )
    
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    
    true_theta = [s[0, 0] for s in true_states]
    est_theta = [s[0, 0] for s in estimates]
    meas_theta = [m[0, 0] for m in meas]
    
    # Estimate plot
    axes[0].plot(times, true_theta, 'b-', lw=2, label='True')
    axes[0].plot(times, est_theta, 'r--', lw=2, label='Gated EKF')
    for idx in outlier_idx:
        axes[0].axvline(times[idx], color='gray', linestyle=':', alpha=0.5)
    axes[0].set_ylabel('Angle (rad)')
    axes[0].set_title(f'Gated EKF (confidence={confidence_level:.3f}, threshold={gate_thresh:.2f})')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # NIS plot
    axes[1].semilogy(times, nis_vals, 'b-', lw=0.5, alpha=0.7, label='NIS')
    axes[1].axhline(gate_thresh, color='red', linestyle='--', lw=2, label=f'Threshold ({gate_thresh:.2f})')
    axes[1].axhline(1.0, color='green', linestyle='-', lw=1, label='Expected mean (1.0)')
    for idx in outlier_idx:
        if idx < len(nis_vals):
            axes[1].scatter(times[idx], nis_vals[idx], c='red', s=100, marker='x', zorder=5)
    axes[1].set_xlabel('Time (s)')
    axes[1].set_ylabel('NIS (log scale)')
    axes[1].set_title('NIS Time Series')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim([0.01, max(1000, max(nis_vals) * 2)])
    
    plt.tight_layout()
    plt.show()
    
    # Statistics
    print(f"Outliers injected: {n_outliers} at indices {outlier_idx}")
    print(f"Measurements rejected: {len(rejected)} ({100*len(rejected)/n_steps:.2f}%)")
    print(f"Correctly rejected outliers: {len(set(outlier_idx) & set(rejected))}/{n_outliers}")
    
    # Check for false rejections (valid measurements rejected)
    false_rejections = len(set(rejected) - set(outlier_idx))
    print(f"False rejections (valid measurements rejected): {false_rejections}")
    
    print("\nTuning guide:")
    print("  - Low confidence -> rejects more (may reject valid measurements)")
    print("  - High confidence -> rejects less (may accept outliers)")
    print("  - Small outliers (< 3 sigma) are hard to detect")

### Outlier Rejection Takeaways

**What we learned:**
1. **NIS is chi-squared distributed** for a consistent filter. Large NIS indicates outlier or model mismatch.
2. **Chi-squared gating** provides a principled threshold. Use `scipy.stats.chi2.ppf(confidence, df=measurement_dim)`.
3. **99% confidence is a good default**. It rejects only 1% of valid measurements while catching most outliers.
4. **Small outliers are hard to detect**. Outliers below ~3 sigma blend into normal noise.

**Practical uses:**
- GPS multipath rejection
- LIDAR outlier rejection
- Any sensor with occasional glitches

## Section 11: NIS-Based Divergence Detection

So far we've used NIS for outlier rejection. But NIS is even more powerful: it can diagnose **why** your EKF is failing.

### NIS as a Filter Health Metric

For a **consistent** (properly tuned) filter, NIS is chi-squared distributed:
- **Degrees of freedom** = measurement dimension
- **Expected mean** = measurement dimension (1 for 1D, 2 for 2D, etc.)

**Diagnostic rules:**
- **Mean(NIS) >> measurement_dim**: Filter is **overconfident** (Q too low, or model mismatch)
- **Mean(NIS) << measurement_dim**: Filter is **underconfident** (Q too high)
- **Mean(NIS) ~ measurement_dim**: Filter is **consistent** (properly tuned)

**Why this works**: NIS measures how surprised the filter is by each measurement. If the filter is overconfident (thinks it knows the state precisely), actual measurements will seem surprisingly large. If underconfident (thinks there's huge uncertainty), measurements will seem surprisingly small.

In [ ]:
def run_ekf_with_nis(measurements, true_states, x0, P0, f, F, Q, h, H, R, dt):
    """
    Run EKF while collecting NIS values and innovations.
    Returns: estimates, covariances, nis_values, innovations
    """
    x_est = x0.copy()
    P_est = P0.copy()
    estimates = []
    covariances = []
    nis_values = []
    innovations = []

    for k, z in enumerate(measurements):
        x_pred, P_pred = ekf_predict(x_est, P_est, f, F, Q, dt)
        nis = compute_nis(z, x_pred, h, H, P_pred, R)
        nis_values.append(nis)
        innovation = z - h(x_pred)
        innovations.append(innovation[0, 0])
        x_est, P_est = ekf_update(x_pred, P_pred, z, h, H, R)
        estimates.append(x_est.copy())
        covariances.append(P_est.copy())
    return estimates, covariances, nis_values, innovations


def diagnose_nis(nis_values, measurement_dim, window_size=50):
    """
    Diagnose filter health from NIS time series.
    Returns: diagnosis string, mean_nis, windowed_mean
    """
    expected_mean = measurement_dim
    mean_nis = np.mean(nis_values)
    windowed_mean = np.convolve(nis_values, np.ones(window_size)/window_size, mode='valid')

    if mean_nis > 1.5 * expected_mean:
        diagnosis = "OVERCONFIDENT: Q likely too low or model mismatch"
    elif mean_nis < 0.5 * expected_mean:
        diagnosis = "UNDERCONFIDENT: Q likely too high"
    else:
        diagnosis = "CONSISTENT: Filter properly tuned"
    return diagnosis, mean_nis, windowed_mean

print("NIS diagnosis functions defined:")
print("  run_ekf_with_nis: Run EKF collecting NIS and innovations")
print("  diagnose_nis: Diagnose filter health from NIS time series")

## Section 12: NIS Diagnosis Demo

Let's see how NIS reveals Q tuning problems. We'll run three EKFs:

1. **Q_good** = diag([0.01, 0.1]) - Well-tuned baseline
2. **Q_low** = diag([0.0001, 0.001]) - Overconfident (trusts model too much)
3. **Q_high** = diag([1.0, 10.0]) - Underconfident (trusts model too little)

In [ ]:
# Define Q matrices for comparison
Q_good = np.diag([0.01, 0.1])      # Well-tuned
Q_low = np.diag([0.0001, 0.001])   # Overconfident (Q too low)
Q_high = np.diag([1.0, 10.0])      # Underconfident (Q too high)

# Initial conditions
x0 = np.array([[0.0], [0.0]])
P0 = np.eye(2) * 0.1

# Run EKF with each Q setting
print("Running EKF with Q_good (well-tuned)...")
est_good, cov_good, nis_good, innov_good = run_ekf_with_nis(
    measurements, true_states, x0, P0,
    pendulum_f, pendulum_F, Q_good,
    measurement_h, measurement_H, R, dt)

print("Running EKF with Q_low (overconfident)...")
est_low, cov_low, nis_low, innov_low = run_ekf_with_nis(
    measurements, true_states, x0, P0,
    pendulum_f, pendulum_F, Q_low,
    measurement_h, measurement_H, R, dt)

print("Running EKF with Q_high (underconfident)...")
est_high, cov_high, nis_high, innov_high = run_ekf_with_nis(
    measurements, true_states, x0, P0,
    pendulum_f, pendulum_F, Q_high,
    measurement_h, measurement_H, R, dt)

print("
All runs complete.")

In [ ]:
# Diagnose each case
measurement_dim = 1

diag_good, mean_good, windowed_good = diagnose_nis(nis_good, measurement_dim)
diag_low, mean_low, windowed_low = diagnose_nis(nis_low, measurement_dim)
diag_high, mean_high, windowed_high = diagnose_nis(nis_high, measurement_dim)

print("NIS Diagnosis Results:")
print("=" * 60)
print(f"Q_good:  Mean NIS = {mean_good:.3f} (expected: {measurement_dim})")
print(f"         {diag_good}")
print()
print(f"Q_low:   Mean NIS = {mean_low:.3f} (expected: {measurement_dim})")
print(f"         {diag_low}")
print()
print(f"Q_high:  Mean NIS = {mean_high:.3f} (expected: {measurement_dim})")
print(f"         {diag_high}")
print("=" * 60)

In [ ]:
# Plot NIS comparison
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

titles = ['Q_good (well-tuned)', 'Q_low (overconfident)', 'Q_high (underconfident)']
nis_data = [nis_good, nis_low, nis_high]
windowed_data = [windowed_good, windowed_low, windowed_high]
means = [mean_good, mean_low, mean_high]
diagnoses = [diag_good, diag_low, diag_high]

chi2_99 = chi2.ppf(0.99, df=measurement_dim)

for i, (title, nis, windowed, mean_nis, diag) in enumerate(zip(titles, nis_data, windowed_data, means, diagnoses)):
    # Row 1: NIS time series
    axes[0, i].semilogy(times, nis, 'b-', lw=0.5, alpha=0.5, label='NIS')
    axes[0, i].axhline(measurement_dim, color='green', linestyle='-', lw=2, label=f'Expected ({measurement_dim})')
    axes[0, i].axhline(chi2_99, color='red', linestyle='--', lw=1, label=f'99% threshold ({chi2_99:.2f})')
    axes[0, i].axhline(mean_nis, color='orange', linestyle='-', lw=2, label=f'Mean ({mean_nis:.2f})')
    axes[0, i].set_ylabel('NIS (log scale)')
    axes[0, i].set_title(title)
    axes[0, i].legend(loc='upper right', fontsize=8)
    axes[0, i].grid(True, alpha=0.3)
    axes[0, i].set_ylim([0.01, 100])

    # Row 2: Windowed mean NIS
    window_times = times[49:]
    axes[1, i].plot(window_times, windowed, 'b-', lw=2, label='Windowed mean')
    axes[1, i].axhline(measurement_dim, color='green', linestyle='-', lw=2, label='Expected')
    axes[1, i].axhline(1.5 * measurement_dim, color='red', linestyle='--', lw=1, label='Overconfident threshold')
    axes[1, i].axhline(0.5 * measurement_dim, color='orange', linestyle='--', lw=1, label='Underconfident threshold')
    axes[1, i].set_xlabel('Time (s)')
    axes[1, i].set_ylabel('Windowed Mean NIS')
    axes[1, i].set_title(f'Diagnosis: {diag.split(":")[0]}')
    axes[1, i].legend(loc='upper right', fontsize=8)
    axes[1, i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### NIS Diagnosis Interpretation

**Q_good (well-tuned):**
- Mean NIS close to 1 (measurement dimension)
- NIS fluctuates around expected value
- Filter is consistent - trusts model and measurements appropriately

**Q_low (overconfident):**
- Mean NIS >> 1
- Filter thinks it knows the state better than it does
- Innovations are larger than expected (filter is "surprised")
- **Fix**: Increase Q to trust the model less

**Q_high (underconfident):**
- Mean NIS << 1
- Filter has huge uncertainty, measurements barely change the estimate
- Innovations are smaller than expected relative to huge covariance
- **Fix**: Decrease Q to trust the model more

**Key insight**: NIS mean should equal measurement dimension. Deviations indicate Q/R miscalibration or model mismatch.

## Section 13: Innovation Whiteness Test

NIS mean tells us about overall filter health. **Innovation autocorrelation** tells us about temporal consistency.

### What is Innovation Whiteness?

For a properly tuned EKF, innovations should be **white noise** (uncorrelated over time). If innovations are correlated:

- **Positive lag-1 autocorrelation**: Innovations persist (filter too sluggish, Q too low)
- **Negative lag-1 autocorrelation**: Innovations flip sign (filter overreacts, Q too high)
- **Autocorrelation within bounds**: Innovations are white (filter properly tuned)

**Why this works**: If the filter is too sluggish (Q low), it doesn't react fast enough to changes, so innovations in the same direction persist. If the filter overreacts (Q high), it overshoots, causing innovations to flip sign.

In [ ]:
def innovation_whiteness_test(innovations, max_lag=20):
    """
    Test if innovation sequence is white (uncorrelated).
    Returns: autocorrelation coefficients, 95% confidence bound
    """
    n = len(innovations)
    innovations = np.array(innovations).flatten()
    mean_innov = np.mean(innovations)
    var_innov = np.var(innovations)

    if var_innov < 1e-10:
        return [1.0] + [0.0] * max_lag, 1.96 / np.sqrt(n)

    autocorr = [1.0]  # lag 0 is always 1
    for lag in range(1, max_lag + 1):
        c = np.mean((innovations[lag:] - mean_innov) * (innovations[:-lag] - mean_innov))
        autocorr.append(c / var_innov)

    # 95% confidence bounds (Bartlett approximation)
    confidence_bound = 1.96 / np.sqrt(n)

    return autocorr, confidence_bound

print("innovation_whiteness_test function defined")
print("  Returns autocorrelation coefficients and 95% confidence bound")
print("  Autocorrelation outside bounds indicates non-white innovations")

In [ ]:
# Compute autocorrelation for each Q setting
autocorr_good, conf_bound = innovation_whiteness_test(innov_good)
autocorr_low, _ = innovation_whiteness_test(innov_low)
autocorr_high, _ = innovation_whiteness_test(innov_high)

print(f"95% confidence bound: +/- {conf_bound:.4f}")
print(f"
Lag-1 autocorrelation:")
print(f"  Q_good: {autocorr_good[1]:.4f} {'(within bounds)' if abs(autocorr_good[1]) < conf_bound else '(OUTSIDE bounds)'}")
print(f"  Q_low:  {autocorr_low[1]:.4f} {'(within bounds)' if abs(autocorr_low[1]) < conf_bound else '(OUTSIDE bounds)'}")
print(f"  Q_high: {autocorr_high[1]:.4f} {'(within bounds)' if abs(autocorr_high[1]) < conf_bound else '(OUTSIDE bounds)'}")

In [ ]:
# Plot autocorrelation comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

titles = ['Q_good (well-tuned)', 'Q_low (overconfident)', 'Q_high (underconfident)']
autocorrs = [autocorr_good, autocorr_low, autocorr_high]

for i, (title, ac) in enumerate(zip(titles, autocorrs)):
    lags = range(len(ac))
    colors = ['green' if abs(a) < conf_bound else 'red' for a in ac]
    colors[0] = 'blue'  # Lag 0 is always 1

    axes[i].bar(lags, ac, color=colors, alpha=0.7, edgecolor='black')
    axes[i].axhline(conf_bound, color='red', linestyle='--', lw=1, label='95% bounds')
    axes[i].axhline(-conf_bound, color='red', linestyle='--', lw=1)
    axes[i].axhline(0, color='black', linestyle='-', lw=0.5)
    axes[i].set_xlabel('Lag')
    axes[i].set_ylabel('Autocorrelation')
    axes[i].set_title(title)
    axes[i].legend(loc='upper right')
    axes[i].grid(True, alpha=0.3, axis='y')
    axes[i].set_ylim([-0.5, 1.1])

plt.tight_layout()
plt.show()

print("
Interpretation:")
print("  Green bars: Within 95% confidence bounds (consistent with white noise)")
print("  Red bars: Outside bounds (innovations are correlated)")
print("  Lag-1 positive (Q_low): Filter too sluggish, innovations persist")
print("  Lag-1 negative (Q_high): Filter overreacts, innovations flip sign")

## Section 14: Complete Diagnostic Dashboard

Now we combine everything into a comprehensive diagnostic dashboard that provides at-a-glance filter health assessment.

In [ ]:
def ekf_diagnostic_dashboard(times, true_states, estimates, covariances,
                              nis_values, innovations, measurement_dim=1):
    """
    Comprehensive EKF diagnostic visualization.

    6-panel figure:
    1. State tracking (estimate vs true)
    2. Estimation error with 2-sigma bounds
    3. NIS time series with thresholds
    4. Innovation autocorrelation
    5. Covariance trace over time
    6. Diagnosis summary (text)
    """
    fig = plt.figure(figsize=(14, 10))

    # Extract data
    true_theta = [s[0, 0] for s in true_states]
    est_theta = [s[0, 0] for s in estimates]
    errors = [est_theta[k] - true_theta[k] for k in range(len(estimates))]
    sigma = [2 * np.sqrt(covariances[k][0, 0]) for k in range(len(covariances))]
    trace_P = [np.trace(P) for P in covariances]

    # Panel 1: State tracking
    ax1 = fig.add_subplot(3, 2, 1)
    ax1.plot(times, true_theta, 'b-', lw=2, label='True')
    ax1.plot(times, est_theta, 'r--', lw=2, label='Estimate')
    ax1.set_ylabel('Angle (rad)')
    ax1.set_title('State Tracking')
    ax1.legend(loc='upper right')
    ax1.grid(True, alpha=0.3)

    # Panel 2: Error with 2-sigma bounds
    ax2 = fig.add_subplot(3, 2, 2)
    ax2.plot(times, errors, 'b-', lw=1, label='Error')
    ax2.fill_between(times, [-s for s in sigma], sigma, color='red', alpha=0.2, label='2-sigma')
    ax2.set_ylabel('Error (rad)')
    ax2.set_title('Estimation Error (should stay in 2-sigma)')
    ax2.legend(loc='upper right')
    ax2.grid(True, alpha=0.3)

    # Panel 3: NIS time series
    ax3 = fig.add_subplot(3, 2, 3)
    ax3.semilogy(times, nis_values, 'b-', lw=0.5, alpha=0.7)
    ax3.axhline(measurement_dim, color='green', linestyle='-', lw=2, label=f'Expected mean ({measurement_dim})')
    ax3.axhline(chi2.ppf(0.99, df=measurement_dim), color='red', linestyle='--', lw=1, label='99% threshold')
    ax3.set_ylabel('NIS (log scale)')
    ax3.set_title('NIS Time Series')
    ax3.legend(loc='upper right')
    ax3.grid(True, alpha=0.3)

    # Panel 4: Innovation autocorrelation
    ax4 = fig.add_subplot(3, 2, 4)
    autocorr, conf_bound = innovation_whiteness_test(innovations)
    lags = range(len(autocorr))
    colors = ['green' if abs(a) < conf_bound else 'red' for a in autocorr]
    colors[0] = 'blue'
    ax4.bar(lags, autocorr, color=colors, alpha=0.7)
    ax4.axhline(conf_bound, color='red', linestyle='--', lw=1, label='95% bounds')
    ax4.axhline(-conf_bound, color='red', linestyle='--', lw=1)
    ax4.set_xlabel('Lag')
    ax4.set_ylabel('Autocorrelation')
    ax4.set_title('Innovation Autocorrelation (should be white)')
    ax4.legend(loc='upper right')
    ax4.grid(True, alpha=0.3, axis='y')

    # Panel 5: Covariance trace
    ax5 = fig.add_subplot(3, 2, 5)
    ax5.semilogy(times, trace_P, 'b-', lw=2)
    ax5.set_xlabel('Time (s)')
    ax5.set_ylabel('trace(P)')
    ax5.set_title('Covariance Trace (should converge)')
    ax5.grid(True, alpha=0.3)

    # Panel 6: Diagnosis summary
    ax6 = fig.add_subplot(3, 2, 6)
    ax6.axis('off')

    # Compute diagnostics
    mean_nis = np.mean(nis_values)
    autocorr_lag1 = autocorr[1] if len(autocorr) > 1 else 0
    final_trace = trace_P[-1] if trace_P else 0

    diagnosis_text = f"""
    DIAGNOSIS SUMMARY
    -----------------
    Mean NIS: {mean_nis:.3f} (expected: {measurement_dim})
    Lag-1 autocorrelation: {autocorr_lag1:.3f}
    Final covariance trace: {final_trace:.6f}

    INTERPRETATION:
    """

    if mean_nis > 1.5 * measurement_dim:
        diagnosis_text += "- NIS HIGH: Filter overconfident, increase Q
"
    elif mean_nis < 0.5 * measurement_dim:
        diagnosis_text += "- NIS LOW: Filter underconfident, decrease Q
"
    else:
        diagnosis_text += "- NIS OK: Filter consistent
"

    if abs(autocorr_lag1) > conf_bound:
        if autocorr_lag1 > 0:
            diagnosis_text += "- POSITIVE AUTOCORR: Q too low, filter sluggish
"
        else:
            diagnosis_text += "- NEGATIVE AUTOCORR: Q too high, filter jittery
"
    else:
        diagnosis_text += "- AUTOCORR OK: Innovations white
"

    ax6.text(0.1, 0.9, diagnosis_text, transform=ax6.transAxes, fontsize=10,
             verticalalignment='top', fontfamily='monospace')

    plt.tight_layout()
    return fig

print("ekf_diagnostic_dashboard function defined")
print("  Creates 6-panel diagnostic visualization")
print("  Includes DIAGNOSIS SUMMARY with actionable guidance")

In [ ]:
# Demo dashboard for each Q setting
print("Diagnostic Dashboard: Q_good (well-tuned)")
fig_good = ekf_diagnostic_dashboard(times, true_states, est_good, cov_good, nis_good, innov_good)
plt.show()

In [ ]:
print("Diagnostic Dashboard: Q_low (overconfident)")
fig_low = ekf_diagnostic_dashboard(times, true_states, est_low, cov_low, nis_low, innov_low)
plt.show()

In [ ]:
print("Diagnostic Dashboard: Q_high (underconfident)")
fig_high = ekf_diagnostic_dashboard(times, true_states, est_high, cov_high, nis_high, innov_high)
plt.show()

### Dashboard Interpretation Guide

**How to use the dashboard:**

1. **Check NIS mean** (Panel 3 + Summary)
   - Should equal measurement dimension
   - High NIS = overconfident = increase Q
   - Low NIS = underconfident = decrease Q

2. **Check innovation autocorrelation** (Panel 4 + Summary)
   - All bars should be within red dashed bounds
   - Positive lag-1 = filter sluggish = increase Q
   - Negative lag-1 = filter jittery = decrease Q

3. **Check covariance trace** (Panel 5)
   - Should converge to steady state
   - Growing = filter diverging
   - Too small = overconfident

4. **Check error bounds** (Panel 2)
   - Errors should stay within 2-sigma bounds
   - Frequent bound violations = filter inconsistent

**Systematic tuning:**
1. Start with larger Q (noisy but stable)
2. Decrease Q until NIS mean ~ measurement_dim
3. Verify autocorrelation is white
4. If NIS stays high with any Q, check model accuracy

## Section 15: Interactive Diagnosis Widget

Explore how Q tuning affects all diagnostic metrics in real-time.

In [ ]:
@interact_manual(
    q_scale=FloatLogSlider(value=1.0, base=10, min=-3, max=2, step=0.25, description='Q scale:')
)
def explore_q_tuning(q_scale):
    """
    Explore Q tuning with diagnostic dashboard.

    Q_scale multiplies the baseline Q:
    - scale < 1: Filter overconfident (trusts model too much)
    - scale = 1: Baseline (well-tuned)
    - scale > 1: Filter underconfident (trusts model too little)
    """
    Q_scaled = Q_good * q_scale

    print(f"Q_scale = {q_scale:.4f}")
    print(f"Q = diag([{Q_scaled[0,0]:.6f}, {Q_scaled[1,1]:.6f}])")
    print()

    # Run EKF with scaled Q
    est, cov, nis, innov = run_ekf_with_nis(
        measurements, true_states, x0, P0,
        pendulum_f, pendulum_F, Q_scaled,
        measurement_h, measurement_H, R, dt
    )

    # Show diagnostic dashboard
    fig = ekf_diagnostic_dashboard(times, true_states, est, cov, nis, innov)
    plt.show()

    print("
Tuning guide:")
    print("  - Increase Q scale if NIS high or positive autocorrelation")
    print("  - Decrease Q scale if NIS low or negative autocorrelation")
    print("  - Target: Mean NIS ~ 1, autocorrelation within bounds")

## Section 16: Key Takeaways

### Sensor Failure Handling

**Dropout (missing measurements):**
- Just skip the update step. Predict continues, covariance grows.
- Longer dropout = more uncertainty = bigger correction when measurements resume.
- No special handling needed -- this is a fundamental Kalman filter property.

**Bias drift:**
- Add bias to state vector (state augmentation).
- Model bias as random walk: bias_new = bias_old, with small process noise.
- EKF estimates bias alongside original states.
- Key requirement: bias must be observable (dynamics distinguish bias from true state).

**Outliers:**
- Compute NIS (Normalized Innovation Squared) before update.
- Reject measurements where NIS > chi-squared threshold.
- Use 99% confidence (not 95%) for conservative rejection.
- Trade-off: too aggressive = rejects valid data, too loose = accepts outliers.

### Divergence Diagnosis

**NIS monitoring:**
- For consistent filter, mean(NIS) = measurement dimension.
- NIS >> expected: filter overconfident (Q too low, model mismatch).
- NIS << expected: filter underconfident (Q too high).

**Innovation whiteness:**
- Properly tuned EKF has white (uncorrelated) innovations.
- Positive lag-1 autocorrelation: filter sluggish (Q too low).
- Negative lag-1 autocorrelation: filter jittery (Q too high).

**Systematic tuning:**
1. Check NIS mean (should equal measurement dimension).
2. Check innovation autocorrelation (should be white).
3. If NIS high + positive autocorr: increase Q.
4. If NIS low + negative autocorr: decrease Q.
5. If NIS consistently high with any Q: check model accuracy.

### When Does the EKF Fail?

- **Model mismatch**: Wrong physics, missing dynamics, incorrect parameters.
- **Non-Gaussian noise**: Heavy-tailed noise, systematic errors.
- **High nonlinearity**: Linearization fails near singularities.
- **Numerical issues**: Covariance loses positive-definiteness (use Joseph form).

### Practical Advice

- Always monitor NIS during development and deployment.
- Verify your model against ground truth before trusting diagnostics.
- Start with larger Q (filter tracks but noisy), then decrease until NIS ~ measurement_dim.
- Use state augmentation for any slowly-varying unknown parameter.